In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Load the CSV dataset
csv_path = "amazon_reviews_us_Apparel_v1_00.csv"
full_df = pd.read_csv(csv_path)

# Select relevant columns
columns_to_use = [
    "star_rating",
    "review_body",
    "helpful_votes",
    "total_votes",
    "verified_purchase",
    "vine"
]
df = full_df[columns_to_use].dropna(subset=["star_rating", "review_body"])

# Sample 50,000 rows stratified by star_rating (max 10,000 per class)
df_sample = df.groupby("star_rating", group_keys=False).apply(
    lambda x: x.sample(min(len(x), 10000), random_state=42)
).reset_index(drop=True)

# Clean review_body text: lowercase and remove special characters
df_sample["review_body"] = df_sample["review_body"].str.lower().str.replace(r'[^a-z0-9\s]', '', regex=True)

# Binary encode 'verified_purchase' and 'vine' columns
df_sample["verified_purchase"] = df_sample["verified_purchase"].map({"Y": 1, "N": 0})
df_sample["vine"] = df_sample["vine"].map({"Y": 1, "N": 0})

# Separate features and target
X = df_sample.drop("star_rating", axis=1)
y = df_sample["star_rating"]

# One-hot encode the star_rating target (5 classes)
y_onehot = OneHotEncoder(sparse=False).fit_transform(y.values.reshape(-1, 1))

# Create preprocessing pipelines
text_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=1000, stop_words="english"))
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", MinMaxScaler())
])

# Combine preprocessing steps
preprocessor = ColumnTransformer([
    ("text", text_pipeline, "review_body"),
    ("num", numeric_pipeline, ["helpful_votes", "total_votes"]),
    ("bin", "passthrough", ["verified_purchase", "vine"])
])

# Transform the data
X_processed = preprocessor.fit_transform(X)

# Final shape of features and labels
print("Input shape:", X_processed.shape)   # (50000, 1004)
print("Target shape:", y_onehot.shape)     # (50000, 5)
